
<h1 id="TTree-%E4%B8%AD%E4%BD%BF%E7%94%A8-vector-%E7%B1%BB%E5%9E%8B%E7%9A%84-Branch">3.8 用 <code>vector</code> 保存 TTree 数据</h1><p>在探测器数据分析中，一个事件中往往会出现多个 hit。若直接使用固定长度数组，或依赖 hit 数目的动态数组保存这些信息，虽然能够完成数据存储，但在后续分析中往往不便于按单个 hit 进行排序、筛选和配对。利用 <code>vector</code>，可以把同一个 hit 的条带编号、能量和时间统一组织起来，再将一个事件中的全部 hit 保存在同一个容器中，从而使数据结构更加清晰，也更便于后续分析处理。</p>
<p>以 <code>data_16C.root</code> 为例，原始 ROOT 文件中既包含固定长度数组，也包含由 hit 数目决定长度的动态数组。固定数组通常按探测器通道展开保存，动态数组则先记录 hit 数目，再分别保存条带编号、能量等信息。对于这类数据，可以进一步整理为结构体与 <code>vector</code> 的组合形式。</p>
<h3 id="%E5%8E%9F%E5%A7%8B%E6%95%B0%E6%8D%AE%E7%BB%93%E6%9E%84">原始数据结构</h3><p>固定数组常写成：</p>
<div class="highlight"><pre><code class="language-cpp">Double_t d1x[32], d2x[32], d3x[32];
Double_t d1y[32], d2y[32], d3y[32];</code></pre><p>可变长 Branch 则由 hit 数控制写入长度；内存中仍保留足够的固定容量，例如一层的 X 面：</p><pre><code class="language-cpp">Int_t d1xhit;
Int_t d1xs[32];
Double_t d1xe[32];
// tree-&gt;Branch("d1xe", d1xe, "d1xe[d1xhit]/D");</code></pre></div>
<p>如果改用 <code>vector</code> 来组织 hit 信息，可以先定义表示单个 hit 的结构体：</p>
<div class="highlight"><pre><code class="language-cpp">struct dssd
{
  Int_t id;
  Double_t e;
  Double_t t = std::numeric_limits&lt;double&gt;::quiet_NaN();
};</code></pre></div>
<p>再分别用 <code>vector&lt;dssd&gt;</code> 保存三层 DSSD 在 x、y 两侧的 hit：</p>
<div class="highlight"><pre><code class="language-cpp">vector&lt;dssd&gt; x1v, x2v, x3v;
vector&lt;dssd&gt; y1v, y2v, y3v;</code></pre></div>
<p>这样，一个事件中某一侧探测器的全部 hit 就被统一保存在一个 <code>vector</code> 中。相比于将条带编号、能量、时间分别放在不同数组中，这种写法更加紧凑，也更符合后续分析的逻辑。</p>
<hr/>
<h3 id="%E4%BB%8E%E6%99%AE%E9%80%9A-ROOT-%E6%96%87%E4%BB%B6%E7%94%9F%E6%88%90%E5%90%AB-vector-%E7%B1%BB%E5%9E%8B-Branch-%E7%9A%84%E6%96%B0%E6%96%87%E4%BB%B6">从普通 ROOT 文件生成含 <code>vector</code> 类型 Branch 的新文件</h3><p>这一部分的目标，是将原始 ROOT 文件中以数组形式保存的 hit 信息，转换为以 <code>vector</code> 类型 Branch 保存的新 ROOT 文件。</p>
<p>通常的流程是：</p>
<ul>
<li>先利用 <code>MakeClass</code> 为原始树生成基础分析框架；</li>
<li>在此基础上补充自己的分析代码；</li>
<li>将数组形式的输入数据整理为 <code>vector</code>，并写入新的 ROOT 文件。</li>
</ul>
<p>程序中一般需要准备如下文件：</p>
<ul>
<li><code>main.cpp</code></li>
<li><code>makefile</code></li>
<li><code>ana.h</code></li>
<li><code>ana.cpp</code></li>
<li><code>Linkdef.h</code></li>
</ul>
<p>其中，<code>main.cpp</code> 负责输入输出文件和树的管理；<code>ana.h</code> 与 <code>ana.cpp</code> 负责具体的数据处理；<code>Linkdef.h</code> 和 <code>makefile</code> 用于生成并编译 ROOT 字典。</p>
<h4 id="main.cpp"><code>main.cpp</code></h4><p><code>main.cpp</code> 负责打开输入文件 <code>strip_arrays_16C.root</code>，读取其中的树，创建输出文件 <code>vec_16C.root</code> 和新的输出树，然后调用分析类中的 <code>Analysis()</code> 完成逐事件处理，最后将输出树写入文件。</p>
<div class="highlight"><pre><code class="language-cpp">#include &lt;TFile.h&gt;
#include &lt;TTree.h&gt;
#include &lt;iostream&gt;
#include "ana.h"
int main(int argc, char** argv) {
    const char* inputName = argc&gt;1 ? argv[1] : "../../data/strip_arrays_16C.root";
    const char* outputName = argc&gt;2 ? argv[2] : "../../vec_16C.root";
    TFile* input = TFile::Open(inputName);
    if (!input || input-&gt;IsZombie()) return 1;
    TTree* tin = input-&gt;Get&lt;TTree&gt;("tree");
    if (!tin) return 1;
    TFile output(outputName,"RECREATE");
    TTree* tout = new TTree("tree","vector branch");
    {
        ana analysis(tin,tout);
        analysis.Analysis();
        std::cout &lt;&lt; "Input=" &lt;&lt; tin-&gt;GetEntries() &lt;&lt; ", output=" &lt;&lt; tout-&gt;GetEntries() &lt;&lt; '\n';
        output.cd();
        tout-&gt;Write();
    }
    // MakeClass 基类的析构函数已释放 input。
    return 0;
}</code></pre></div>
<h4 id="ana.h"><code>ana.h</code></h4><p>在 <code>ana.h</code> 中，可以定义分析类 <code>ana</code>。它继承自 <code>MakeClass</code> 生成的基类，并声明六个 <code>vector&lt;dssd&gt;</code> 变量，用于分别保存三层 DSSD 在 x、y 两个方向上的 hit。同时，还定义设置输出树、处理单侧探测器数据以及主分析循环所需的成员函数。</p>
<div class="highlight"><pre><code class="language-cpp">#ifndef ana_h
#define ana_h

#include &lt;vector&gt;
#include &lt;limits&gt;
#include &lt;iostream&gt;
#include "test.h"  //包含基类头文件

using namespace std;

struct dssd
{
  Int_t id;
  Double_t e;
  Double_t t = std::numeric_limits&lt;double&gt;::quiet_NaN();
};

class ana : public test //从test类中继承其成员变量和成员函数
{
 public:
  vector&lt;dssd&gt; x1v,x2v,x3v;
  vector&lt;dssd&gt; y1v,y2v,y3v;
  Long64_t source_entry = 0;
  TTree *opt;


 ana(TTree* ipt_,TTree *opt_): test(ipt_),opt(opt_) {}
  virtual ~ana() {};
  virtual void     Analysis();//分析函数，作用等价于原Loop函数
  virtual void     SetOutBranch();
  virtual void     ProcessDS(const Double_t ee[32], vector&lt;dssd&gt; &amp;vec);

};
#endif</code></pre></div>
<h4 id="ana.cpp"><code>ana.cpp</code></h4><p><code>ana.cpp</code> 中主要包括三个部分：</p>
<ul>
<li><code>SetOutBranch()</code>：设置输出树中的 Branch；</li>
<li><code>ProcessDS()</code>：将数组形式的数据整理为 <code>vector</code>；</li>
<li><code>Analysis()</code>：逐事件读取输入树，并完成转换后写入输出树。</li>
</ul>
<div class="highlight"><pre><code class="language-cpp">#include "ana.h"
using namespace std;
void ana::SetOutBranch()
{
  opt-&gt;Branch("source_entry", &amp;source_entry, "source_entry/L");
  opt-&gt;Branch("x1v",&amp;x1v);
  opt-&gt;Branch("x2v",&amp;x2v);
  opt-&gt;Branch("x3v",&amp;x3v);
  opt-&gt;Branch("y1v",&amp;y1v);
  opt-&gt;Branch("y2v",&amp;y2v);
  opt-&gt;Branch("y3v",&amp;y3v);
  opt-&gt;Branch("sx1e",&amp;sx1e,"sx1e/D");
  opt-&gt;Branch("sx2e",&amp;sx2e,"sx2e/D");
  opt-&gt;Branch("sx3e",&amp;sx3e,"sx3e/D");
  opt-&gt;Branch("sy1e",&amp;sy1e,"sy1e/D");
  opt-&gt;Branch("sy2e",&amp;sy2e,"sy2e/D");
  opt-&gt;Branch("sy3e",&amp;sy3e,"sy3e/D");

}

void ana::ProcessDS(const Double_t ee[32], vector&lt;dssd&gt; &amp;vec)
{
    vec.clear(); // 每个事件重新建立 hit 列表
    for(int i=0; i&lt;32; ++i) {
        if(ee[i]&lt;1) continue;
        dssd hit;
        hit.id=i;
        hit.e=ee[i];
        vec.push_back(hit); // t 保持 NaN，输入没有时间信息
    }
}
void ana::Analysis()
{
  if (fChain == 0) return;
  SetOutBranch();
  Long64_t nentries = fChain-&gt;GetEntriesFast();
  for (Long64_t jentry=0; jentry&lt;nentries;jentry++) {
    Long64_t ientry = LoadTree(jentry);
    if (ientry &lt; 0) break;
    fChain-&gt;GetEntry(jentry);
    ProcessDS(d1x,x1v);
    ProcessDS(d1y,y1v);
    ProcessDS(d2x,x2v);
    ProcessDS(d2y,y2v);
    ProcessDS(d3x,x3v);
    ProcessDS(d3y,y3v);
    source_entry = jentry;
    opt-&gt;Fill(); // 保留空事件及事件对应关系


  }
}</code></pre></div>
<p>这里的做法是：对每一个事件，分别把 <code>d1x</code>、<code>d1y</code>、<code>d2x</code>、<code>d2y</code>、<code>d3x</code>、<code>d3y</code> 中满足条件的通道转成 <code>dssd</code> 结构体，再压入对应的 <code>vector</code> 中。每个输入事件都写入输出树，包括空 vector；这样后续仍能追溯原事件。</p>
<hr/>
<h3 id="%E5%AD%97%E5%85%B8%E4%B8%8E%E7%BC%96%E8%AF%91">字典与编译</h3><p>当 TTree 的 Branch 中保存的是自定义结构体，或 <code>vector&lt;自定义类型&gt;</code> 这类 STL 容器时，需要为相应类型生成 ROOT 字典。否则，ROOT 在读取文件时将无法正确识别这些类型，并可能给出找不到字典的警告信息，从而影响数据的正常读写与后续分析。</p>
<p>例如，在本节中，输出 Branch 中使用了自定义结构体 <code>dssd</code> 以及 <code>vector&lt;dssd&gt;</code>，因此需要在 <code>Linkdef.h</code> 中加入相应的字典声明：</p>
<div class="highlight"><pre><code class="language-cpp">#ifdef __CLING__

#pragma link off all globals;
#pragma link off all classes;
#pragma link off all functions;
#pragma link C++ nestedclasses;

#pragma link C++ class dssd+;
#pragma link C++ class vector&lt;dssd&gt;+;

#endif</code></pre></div>
<p>在完成 <code>Linkdef.h</code> 的设置后，还需要在编译过程中调用 <code>rootcling</code> 生成字典源文件，并将该文件与主程序及其他源文件一起编译。为此，可以编写如下 <code>makefile</code>：</p>
<div class="highlight"><pre><code class="language-cpp">CXX = c++
CPPFLAGS = -Iinclude $(shell root-config --cflags)
CXXFLAGS = -O2 -Wall
LDLIBS = $(shell root-config --libs)
SOURCES = main.cpp $(wildcard src/*.cpp src/*.C) LinkDict.cc
HEADERS = $(wildcard include/*.h)

all: dssd libhits.so

LinkDict.cc: $(HEADERS) Linkdef.h
	rootcling -f $@ -Iinclude include/ana.h Linkdef.h

dssd: $(SOURCES) $(HEADERS)
	$(CXX) $(CPPFLAGS) $(CXXFLAGS) $(SOURCES) $(LDLIBS) -o $@

libhits.so: LinkDict.cc $(HEADERS)
	$(CXX) $(CPPFLAGS) $(CXXFLAGS) -fPIC -shared LinkDict.cc $(LDLIBS) -o $@

clean:
	rm -f libhits.so dssd LinkDict.cc LinkDict_rdict.pcm</code></pre></div>
<p>Makefile 中，<code>SOURCES</code> 收集主程序、分析源文件和生成的 LinkDict.cc；<code>HEADERS</code> 作为依赖。<code>CPPFLAGS</code>、<code>CXXFLAGS</code>、<code>LDLIBS</code> 分别提供头文件路径、编译选项与链接库。它同时构建独立程序 dssd 和供 ROOT 会话载入的 libhits.so。</p>
<ul>
<li><code>all</code> 构建 dssd 和 libhits.so；<code>clean</code> 只清理本工程的构建产物。</li>
<li><code>clean</code> 目标用于清除编译过程中产生的中间文件和可执行文件。</li>
</ul>
<p>这里需要特别注意的是，在调用 <code>rootcling</code> 生成字典时，必须将包含相关类型定义的头文件放在 <code>Linkdef.h</code> 之前。例如本例中使用了 <code>include/ana.h</code>，其中定义了 <code>dssd</code> 以及相关 <code>vector</code> 类型，因此应写成：</p>
<div class="highlight"><pre><code class="language-cpp">LinkDict.cc: $(HEADERS) Linkdef.h
	rootcling -f $@ -Iinclude include/ana.h Linkdef.h</code></pre></div>
<p>如果顺序不正确，<code>rootcling</code> 在处理 <code>Linkdef.h</code> 时将无法识别这些类型，从而导致字典生成失败。也就是说，<code>Linkdef.h</code> 只负责声明需要生成字典的类型，而这些类型本身必须在它之前已经被包含并定义。</p>
<hr/>
<h3 id="%E5%9C%A8-ROOT-%E5%91%BD%E4%BB%A4%E8%A1%8C%E4%B8%AD%E4%BD%BF%E7%94%A8%E5%90%AB-vector-%E7%B1%BB%E5%9E%8B-Branch-%E7%9A%84%E6%96%87%E4%BB%B6">在 ROOT 命令行中使用含 <code>vector</code> 类型 Branch 的文件</h3><p>生成含 <code>vector</code> Branch 的新文件后，可以直接在 ROOT 命令行中进行查看和分析。</p>
<p>例如：</p>
<p>本节输入为 <code>data/strip_arrays_16C.root</code>，保留刻度后的固定数组，区别于 3.6 的 compact-hit 文件。它没有时间数据，结构体中的 t 显式设为 NaN，不生成虚构时间。两个转换程序均保留原事件编号和空事件。</p>

In [1]:
%jsroot on


<h3 id="%E5%9C%A8-ROOT-%E5%91%BD%E4%BB%A4%E8%A1%8C%E4%B8%AD%E4%BD%BF%E7%94%A8%E5%90%AB-vector-%E7%B1%BB%E5%9E%8B-Branch-%E7%9A%84%E6%96%87%E4%BB%B6">在 ROOT 命令行中使用含 <code>vector</code> 类型 Branch 的文件</h3><p>生成含 <code>vector</code> Branch 的新文件后，可以直接在 ROOT 命令行中进行查看和分析。</p>
<p>例如：</p>


In [2]:
gSystem->Load("code/code1/libhits.so"); // 本节 make 同时生成读取字典
TCanvas *c1 = new TCanvas;
TFile *ff = new TFile("vec_16C.root");
if (!ff || ff->IsZombie()) throw std::runtime_error("无法打开输入 ROOT 文件");
TTree *tree = (TTree*)ff->Get("tree");
if (!tree) throw std::runtime_error("输入文件中缺少 tree");


<p>如果字典没有正确生成，ROOT 在打开文件时可能会给出类似如下的警告：</p>
<div class="highlight"><pre><span></span><span class="n">Warning</span><span class="w"> </span><span class="n">in</span><span class="w"> </span><span class="o">&lt;</span><span class="n">TClass</span><span class="o">::</span><span class="n">Init</span><span class="o">&gt;:</span><span class="w"> </span><span class="n">no</span><span class="w"> </span><span class="n">dictionary</span><span class="w"> </span><span class="k">for</span><span class="w"> </span><span class="k">class</span><span class="w"> </span><span class="nc">dssd</span><span class="w"> </span><span class="n">is</span><span class="w"> </span><span class="n">available</span>
</pre></div>
<p>这说明 ROOT 虽然能够打开文件，但对自定义类型的支持并不完整，因此前面的字典生成步骤是必要的。</p>
<h4 id="%E6%9F%A5%E7%9C%8B-vector-%E7%9A%84%E9%95%BF%E5%BA%A6">查看 <code>vector</code> 的长度</h4><p>ROOT 提供了 <code>@vec.size()</code> 的写法，用于获得某个 <code>vector</code> 的长度。例如：</p>


In [3]:
tree->Draw("@x1v.size()");//显示vector的大小
c1->Draw();


<p>该语句可以直接画出 <code>x1v</code> 在各事件中的 hit 数分布。</p>
<h4 id="%E8%AE%BF%E9%97%AE-vector-%E4%B8%AD%E5%8D%95%E4%B8%AA%E5%85%83%E7%B4%A0%E7%9A%84%E6%88%90%E5%91%98">访问 <code>vector</code> 中单个元素的成员</h4><p>可以通过</p>
<div class="highlight"><pre><span></span><span class="n">vec</span><span class="p">[</span><span class="n">i</span><span class="p">].</span><span class="n">member</span>
</pre></div>
<p>的形式访问 <code>vector</code> 中第 <code>i</code> 个元素的成员变量。例如：</p>


In [4]:
tree->Scan("x1v[0].id:x1v[0].e:x1v[0].t:x1v[1].id:x1v[1].e:x1v[1].t",
           "@x1v.size()==2", "", 10, 1);

************************************************************************************
*    Row   * x1v[0].id *  x1v[0].e *  x1v[0].t * x1v[1].id *  x1v[1].e *  x1v[1].t *
************************************************************************************
*        1 *        17 * 4925.0811 *       nan *        18 * 1007.1224 *       nan *
*        2 *        23 * 3146.8572 *       nan *        24 * 339.14570 *       nan *
*        3 *        23 * 5465.0851 *       nan *        24 * 41.435740 *       nan *
*        4 *        21 * 5993.6417 *       nan *        22 * 43.867701 *       nan *
*        5 *        11 * 4196.0801 *       nan *        12 * 1159.4884 *       nan *
*        6 *        13 * 387.77402 *       nan *        14 * 3889.7610 *       nan *
************************************************************************************
==> 6 selected entries



<p>表示查看 <code>x1v</code> 中前两个 hit 的条带编号、能量和时间。</p>
<h4 id="%E5%B1%95%E5%BC%80%E6%9F%A5%E7%9C%8B%E5%85%A8%E9%83%A8-hit-%E4%BF%A1%E6%81%AF">展开查看全部 hit 信息</h4><p>除了按下标逐项访问之外，也可以直接写成：</p>


In [5]:
tree->Scan("x1v.id:x1v.e:x1v.t", "@x1v.size()==2", "", 10, 1);

***********************************************************
*    Row   * Instance *    x1v.id *     x1v.e *     x1v.t *
***********************************************************
*        1 *        0 *        17 * 4925.0811 *       nan *
*        1 *        1 *        18 * 1007.1224 *       nan *
*        2 *        0 *        23 * 3146.8572 *       nan *
*        2 *        1 *        24 * 339.14570 *       nan *
*        3 *        0 *        23 * 5465.0851 *       nan *
*        3 *        1 *        24 * 41.435740 *       nan *
*        4 *        0 *        21 * 5993.6417 *       nan *
*        4 *        1 *        22 * 43.867701 *       nan *
*        5 *        0 *        11 * 4196.0801 *       nan *
*        5 *        1 *        12 * 1159.4884 *       nan *
*        6 *        0 *        13 * 387.77402 *       nan *
*        6 *        1 *        14 * 3889.7610 *       nan *
***********************************************************
==> 12 selected entries



<p>在这种情况下，ROOT 会自动把 <code>vector</code> 中的每个元素展开显示，并为每个元素给出对应的 <code>Instance</code> 编号。这样可以方便地查看一个事件中所有 hit 的详细信息。</p>
<h3 id="%E5%9C%A8%E5%90%8E%E7%BB%AD%E5%88%86%E6%9E%90%E4%B8%AD%E7%BB%A7%E7%BB%AD%E4%BF%9D%E6%8C%81-vector-%E7%B1%BB%E5%9E%8B-Branch">在后续分析中继续保持 <code>vector</code> 类型 Branch</h3><p>如果已经生成了含 <code>vector</code> Branch 的 ROOT 文件，后续分析时往往希望继续以 <code>vector</code> 的形式读取这些数据，而不是重新退回到数组形式。此时需要特别注意：不能直接沿用 <code>MakeClass</code> 的默认处理方式。</p>
<p>对本例中 split 的 <code>vector&lt;dssd&gt;</code> Branch，MakeClass 会按长度与成员数组生成读取代码。这种方式可以访问数据，但不直接提供整个 vector 的容器操作。需要继续按 vector 读入时，可用下面的对象指针绑定方式。</p>


In [6]:
tree->Print();

******************************************************************************
*Tree    :tree      : vector branch                                          *
*Entries :  1926502 : Total =       825074564 bytes  File  Size =  342577290 *
*        :          : Tree compression factor =   2.41                       *
******************************************************************************
*Br    0 :source_entry : source_entry/L                                      *
*Entries :  1926502 : Total  Size=   15417703 bytes  File Size  =    2963374 *
*Baskets :       54 : Basket Size=    1520128 bytes  Compression=   5.20     *
*............................................................................*
*Br    1 :x1v       : Int_t x1v_                                             *
*Entries :  1926502 : Total  Size=   15507888 bytes  File Size  =    3614131 *
*Baskets :      735 : Basket Size=      32000 bytes  Compression=   4.28     *
*...................................................


<hr/>
<h3 id="%E5%9C%A8%E5%90%8E%E7%BB%AD%E5%88%86%E6%9E%90%E4%B8%AD%E7%BB%A7%E7%BB%AD%E4%BF%9D%E6%8C%81-vector-%E7%B1%BB%E5%9E%8B-Branch">在后续分析中继续保持 <code>vector</code> 类型 Branch</h3><p>如果已经生成了含 <code>vector</code> Branch 的 ROOT 文件，后续分析时往往希望继续以 <code>vector</code> 的形式读取这些数据，而不是重新退回到数组形式。此时需要特别注意：不能直接沿用 <code>MakeClass</code> 的默认处理方式。</p>
<p>对本例中 split 的 <code>vector&lt;dssd&gt;</code> Branch，MakeClass 会按长度与成员数组生成读取代码。这种方式可以访问数据，但不直接提供整个 vector 的容器操作。需要继续按 vector 读入时，可用下面的对象指针绑定方式。</p>
<div class="highlight"><pre><code class="language-cpp">tree-&gt;Print();</code></pre></div>
<p>可以看到 <code>x1v</code> 之类的 Branch 被展开为：</p>
<ul>
<li><code>x1v_</code></li>
<li><code>x1v.id[x1v_]</code></li>
<li><code>x1v.e[x1v_]</code></li>
<li><code>x1v.t[x1v_]</code></li>
</ul>
<p>这种展开方式虽然可以访问数据，但不利于继续使用 STL 中针对 <code>vector</code> 的各种操作。</p>
<p>因此，若希望在分析代码中继续把输入量视为 <code>vector&lt;dssd&gt;</code>，就需要手动定义指针，并使用 <code>SetBranchAddress()</code> 建立关联。</p>
<h4 id="%E8%BE%93%E5%85%A5-Branch-%E7%9A%84%E5%AE%9A%E4%B9%89">输入 Branch 的定义</h4><p>在新的分析类中，可以将输入 <code>vector</code> Branch 定义为指针，同时定义一个新的结构体用于保存 x-y 配对后的结果：</p>
<div class="highlight"><pre><code class="language-cpp">#ifndef ana_h
#define ana_h

#include &lt;vector&gt;
#include &lt;limits&gt;
#include &lt;algorithm&gt;
#include &lt;TFile.h&gt;
#include &lt;TTree.h&gt;

using namespace std;
struct dssd//aside
{
  Int_t id;
  Double_t e;
  Double_t t = std::numeric_limits&lt;double&gt;::quiet_NaN();
};

struct DSSD//x-y side
{
 int xid;
 int yid;
 double xe; // X 面幅度
 double ye; // Y 面幅度
};

class ana
{
 public:
  vector&lt;dssd&gt; *br_x1v, *br_x2v, *br_x3v;  //声明vector指针
  vector&lt;dssd&gt; *br_y1v, *br_y2v, *br_y3v;
  Double_t sx1e,sx2e,sx3e;//sum，与输入 tree 的 /D 一致
  Double_t sy1e,sy2e,sy3e;
  TTree *ipt;
  Long64_t source_entry = 0;
  TTree *opt;

  vector&lt;DSSD&gt; d1,d2,d3; //output

 ana(TTree* ipt_,TTree *opt_): ipt(ipt_),opt(opt_) {}
  virtual ~ana() {};
  virtual void     SetBranchInput();
  virtual void     GetDSSD(vector&lt;dssd&gt; *x, vector&lt;dssd&gt; *y, vector&lt;DSSD&gt; &amp;xy);
  virtual void     Analysis();
  virtual void     BranchOutput();
};
#endif</code></pre></div>
<h4 id="%E8%AE%BE%E7%BD%AE%E8%BE%93%E5%85%A5-Branch-%E5%9C%B0%E5%9D%80">设置输入 Branch 地址</h4><p>在使用 <code>SetBranchAddress()</code> 之前，应先将这些指针初始化为空：</p>
<div class="highlight"><pre><code class="language-cpp">void ana::SetBranchInput()
{
  ipt-&gt;SetBranchAddress("source_entry", &amp;source_entry);
  br_x1v = nullptr; // ROOT 读入后使指针指向对应的 vector
  br_x2v = nullptr;
  br_x3v = nullptr;
  br_y1v = nullptr;
  br_y2v = nullptr;
  br_y3v = nullptr;
  ipt-&gt;SetBranchAddress("x1v", &amp;br_x1v); //将变量指向对应Branch的地址
  ipt-&gt;SetBranchAddress("x2v", &amp;br_x2v);
  ipt-&gt;SetBranchAddress("x3v", &amp;br_x3v);
  ipt-&gt;SetBranchAddress("y1v", &amp;br_y1v);
  ipt-&gt;SetBranchAddress("y2v", &amp;br_y2v);
  ipt-&gt;SetBranchAddress("y3v", &amp;br_y3v);
  ipt-&gt;SetBranchAddress("sx1e", &amp;sx1e);
  ipt-&gt;SetBranchAddress("sx2e", &amp;sx2e);
  ipt-&gt;SetBranchAddress("sx3e", &amp;sx3e);
  ipt-&gt;SetBranchAddress("sy1e", &amp;sy1e);
  ipt-&gt;SetBranchAddress("sy2e", &amp;sy2e);
  ipt-&gt;SetBranchAddress("sy3e", &amp;sy3e);
}</code></pre></div>
<p>这一步非常重要。若不先初始化为空指针，程序在运行时可能会出现错误。</p>
<h4 id="%E8%AE%BE%E7%BD%AE%E8%BE%93%E5%87%BA-Branch">设置输出 Branch</h4><p>输出时，可以将配对后的结果保存为新的 <code>vector&lt;DSSD&gt;</code> Branch：</p>
<div class="highlight"><pre><code class="language-cpp">void ana::BranchOutput()
{
  opt-&gt;Branch("source_entry", &amp;source_entry, "source_entry/L");
  opt-&gt;Branch("d1",&amp;d1);
  opt-&gt;Branch("d2",&amp;d2);
  opt-&gt;Branch("d3",&amp;d3);
}</code></pre></div>
<h4 id="%E6%8E%92%E5%BA%8F%E4%B8%8E%E9%85%8D%E5%AF%B9">排序与配对</h4><p>由于同一侧探测器可能存在多个 hit，在进行 x-y 关联之前，常常需要先对 hit 按能量进行排序。例如，可以定义一个比较函数：</p>
<div class="highlight"><pre><code class="language-cpp">bool SortDS(const dssd &amp;a, const dssd &amp;b)
{
  return a.e &gt; b.e;
}</code></pre></div>
<p>随后对各 <code>vector</code> 分别排序：</p>
<div class="highlight"><pre><code class="language-cpp">sort(br_x1v-&gt;begin(), br_x1v-&gt;end(), SortDS);
sort(br_y1v-&gt;begin(), br_y1v-&gt;end(), SortDS);</code></pre></div>
<p>在完成排序后，就可以进行 x-y 配对。一个简单的思路是：</p>
<ul>
<li>取 x、y 两侧 hit 数目的较小值作为循环上限；</li>
<li>分别取出对应 hit 的条带编号和能量；</li>
<li>判断 x、y 两侧能量是否匹配；</li>
<li>若满足条件，则构造一个 <code>DSSD</code> 结果并保存。</li>
</ul>
<p>示意代码如下：</p>
<div class="highlight"><pre><code class="language-cpp">void ana::GetDSSD(vector&lt;dssd&gt; *x, vector&lt;dssd&gt; *y, vector&lt;DSSD&gt; &amp;xy)
{
    xy.clear();
    const size_t nPairs=std::min(x-&gt;size(),y-&gt;size());
    for(size_t i=0; i&lt;nPairs; ++i) {
        const dssd &amp;xhit=(*x)[i]; // 引用完整 hit，条号和幅度保持对应
        const dssd &amp;yhit=(*y)[i];
        if(std::abs(xhit.e-yhit.e)&lt;50) {
            DSSD pair;
            pair.xid=xhit.id;
            pair.yid=yhit.id;
            pair.xe=xhit.e;
            pair.ye=yhit.e;
            xy.push_back(pair);
        }
    }
}</code></pre></div>
<p>这里的能量匹配条件、配对策略以及排序方式都可以根据具体实验数据进行调整。</p>
<h4 id="%E5%90%8E%E7%BB%AD%E5%88%86%E6%9E%90%E4%B8%BB%E5%BE%AA%E7%8E%AF">后续分析主循环</h4><p>在 <code>Analysis()</code> 中，完整流程通常包括：</p>
<ul>
<li>设置输入 Branch；</li>
<li>设置输出 Branch；</li>
<li>逐事件读取数据；</li>
<li>对各层探测器的 x、y hit 分别排序；</li>
<li>进行 x-y 配对；</li>
<li>将得到的结果写入新的输出树。</li>
</ul>
<div class="highlight"><pre><code class="language-cpp">void ana::Analysis()
{
  if (ipt == 0) return;
  SetBranchInput();
  BranchOutput();
  Long64_t nentries = ipt-&gt;GetEntriesFast();
  for (Long64_t jentry=0; jentry&lt;nentries;jentry++) {
    ipt-&gt;GetEntry(jentry);
    sort(br_x1v-&gt;begin(),br_x1v-&gt;end(),SortDS);
    sort(br_y1v-&gt;begin(),br_y1v-&gt;end(),SortDS);
    sort(br_x2v-&gt;begin(),br_x2v-&gt;end(),SortDS);
    sort(br_y2v-&gt;begin(),br_y2v-&gt;end(),SortDS);
    sort(br_x3v-&gt;begin(),br_x3v-&gt;end(),SortDS);
    sort(br_y3v-&gt;begin(),br_y3v-&gt;end(),SortDS);
    GetDSSD(br_x1v,br_y1v,d1);
    GetDSSD(br_x2v,br_y2v,d2);
    GetDSSD(br_x3v,br_y3v,d3);
    opt-&gt;Fill(); // 无候选时保存空 vector，不改变事件顺序


  }
}</code></pre></div>
<p>后续分析时的 <code>main.cpp</code> 与前面类似，只是输入文件变为 <code>vec_16C.root</code>，输出文件变为 <code>sort_16C.root</code>：</p>
<div class="highlight"><pre><code class="language-cpp">#include &lt;TFile.h&gt;
#include &lt;TTree.h&gt;
#include &lt;iostream&gt;
#include "ana.h"
int main(int argc, char** argv) {
    const char* inputName = argc&gt;1 ? argv[1] : "../../vec_16C.root";
    const char* outputName = argc&gt;2 ? argv[2] : "../../sort_16C.root";
    TFile* input = TFile::Open(inputName);
    if (!input || input-&gt;IsZombie()) return 1;
    TTree* tin = input-&gt;Get&lt;TTree&gt;("tree");
    if (!tin) return 1;
    TFile output(outputName,"RECREATE");
    TTree* tout = new TTree("tree","vector branch");
    {
        ana analysis(tin,tout);
        analysis.Analysis();
        std::cout &lt;&lt; "Input=" &lt;&lt; tin-&gt;GetEntries() &lt;&lt; ", output=" &lt;&lt; tout-&gt;GetEntries() &lt;&lt; '\n';
        output.cd();
        tout-&gt;Write();
    }
    delete input;
    return 0;
}</code></pre></div>
<p>如果输出中又引入了新的结构体 <code>DSSD</code> 以及 <code>vector&lt;DSSD&gt;</code>，那么也需要继续在 <code>Linkdef.h</code> 中补充对应的字典声明：</p>
<div class="highlight"><pre><code class="language-cpp">#ifdef __CLING__

#pragma link off all globals;
#pragma link off all classes;
#pragma link off all functions;
#pragma link C++ nestedclasses;

#pragma link C++ class dssd+;
#pragma link C++ class vector&lt;dssd&gt;+;
#pragma link C++ class DSSD+;
#pragma link C++ class vector&lt;DSSD&gt;+;

#endif</code></pre></div>
<hr/>
<h2 id="%E5%AE%9E%E9%99%85%E4%BB%A3%E7%A0%81">实际代码</h2><p>本节对应的完整可运行代码放在 GitHub 仓库的 <code>chapt3/code</code> 目录下，其中包含 <code>code1</code> 和 <code>code2</code> 两个子目录。<code>code1</code> 的主程序读取 <code>strip_arrays_16C.root</code>，输出 <code>vec_16C.root</code>；<code>code2</code> 的主程序读取 <code>vec_16C.root</code>，输出 <code>sort_16C.root</code>。这两个目录对应前后两个连续的分析步骤。<br/>
代码目录：<br/>
<a href="https://github.com/zhihuanli/Experimental-Data-Analysis-Course/tree/master/chapt3/code">https://github.com/zhihuanli/Experimental-Data-Analysis-Course/tree/master/chapt3/code</a> (<a href="https://github.com/zhihuanli/Experimental-Data-Analysis-Course/tree/master/chapt3/code">GitHub</a>)</p>
<p><code>code1</code> 用于将原始 ROOT 文件中的数组形式数据整理成 <code>vector</code> 类型的 Branch，并写入新的 ROOT 文件。对应讲义中“从普通 ROOT 文件生成含 <code>vector</code> 类型 Branch 的新文件”这一部分。<code>code1</code> 中的 <code>main.cpp</code> 明确给出了输入文件 <code>strip_arrays_16C.root</code> 和输出文件 <code>vec_16C.root</code>，而 <code>src/ana.cpp</code> 中则完成了 <code>x1v</code>、<code>x2v</code>、<code>x3v</code>、<code>y1v</code>、<code>y2v</code>、<code>y3v</code> 等 <code>vector</code> Branch 的建立与填充。</p>
<p><code>code2</code> 用于继续读取已经包含 <code>vector</code> Branch 的 ROOT 文件，在分析中保持 <code>vector</code> 类型读入，完成 x-y hit 的排序、配对，并将结果写入新的 ROOT 文件。对应讲义中“使用含 <code>vector</code> 类型 Branch 的 ROOT 文件继续进行分析”这一部分。<code>code2</code> 的 <code>main.cpp</code> 给出了输入文件 <code>vec_16C.root</code> 和输出文件 <code>sort_16C.root</code>，而 <code>src/ana.cpp</code> 中则通过 <code>SetBranchAddress()</code> 读取 <code>vector&lt;dssd&gt;</code>，并将配对结果输出为 <code>d1</code>、<code>d2</code>、<code>d3</code> 等新的 Branch。</p>
<h3 id="vector-%E7%9A%84%E5%B8%B8%E7%94%A8%E6%93%8D%E4%BD%9C"><code>vector</code> 的常用操作</h3><p>在实际分析中，<code>vector</code> 的优势不仅在于可以保存变长数据，更重要的是可以直接利用 STL 提供的各种操作完成排序、筛选和去重等任务。</p>
<p>常见的 <code>vector</code> 成员函数包括：</p>
<div class="highlight"><pre><code class="language-cpp">x1v.size();             // 当前 hit 数
x1v.clear();            // 清空当前事件的 hit
x1v.push_back(ds);       // 加入一个 dssd 元素
x1v.begin();             // 指向第一个元素的迭代器
x1v.end();               // 指向末尾之后的位置，不能解引用
x1v.assign(x2v.begin(), x2v.end()); // 复制另一个 vector 的内容
// 若 it 指向有效元素，erase 返回删除位置之后的迭代器：
// it = x1v.erase(it);</code></pre></div>
<h4 id="%E6%8E%92%E5%BA%8F">排序</h4><p>若要按能量从高到低排序，可结合比较函数与 <code>sort()</code> 使用：</p>
<div class="highlight"><pre><code class="language-cpp">bool SortDS(const dssd &amp;a, const dssd &amp;b)
{
  return a.e &gt; b.e;
}
sort(x1v.begin(), x1v.end(), SortDS);</code></pre></div>
<h4 id="%E5%88%A0%E9%99%A4%E9%87%8D%E5%A4%8D%E5%85%83%E7%B4%A0">删除重复元素</h4><p>如果需要删除重复 hit，可以先定义“重复”的判据，再结合 <code>unique()</code> 与 <code>erase()</code> 使用：</p>
<div class="highlight"><pre><code class="language-cpp">bool Equal(dssd &amp;a, dssd &amp;b)
{
  return a.id == b.id &amp;&amp; a.e == b.e &amp;&amp; a.t == b.t;
}

void Unique(vector&lt;dssd&gt; &amp;a)
{
  a.erase(unique(a.begin(), a.end(), Equal), a.end());
}

// 仅演示删除相邻的等价元素；先独立确认确为重复记录
Unique(xvec);</code></pre></div>
<h4 id="%E6%8C%89%E6%9D%A1%E4%BB%B6%E5%88%A0%E9%99%A4%E5%85%83%E7%B4%A0">按条件删除元素</h4><p>在很多分析中，需要从 <code>vector</code> 中删除不满足条件的元素。例如，当某个 hit 的时间与参考时间相差过大时，可以将其剔除：</p>
<div class="highlight"><pre><code class="language-cpp">void tCut(vector&lt;dssd&gt; &amp;a, vector&lt;dssd&gt; &amp;b, double t1, double t2)
{
  if (b.size() &gt; 0) {
    const double referenceTime = b[0].t; // 删除元素前保存参考时间
    for (auto it = a.begin(); it != a.end(); ) {
      double dt = it-&gt;t - referenceTime;
      if (dt &lt; t1 || dt &gt; t2)
        it = a.erase(it);
      else
        ++it;
    }
  }
}

tCut(x1v, x1v, -20, 20);</code></pre></div>
<p>这些例子说明，使用 <code>vector</code> 的优势不仅在于能够保存变长 hit 信息，还在于可以直接利用 STL 提供的排序、去重和筛选等操作，使后续分析实现起来更加自然。</p>
<p><code>unique</code> 只去掉相邻的等价元素；相同条号和近似相等的能量不能证明是重复读出，真实 pileup 也可能如此。数据去重应有原始事件标识、timestamp 或电子学重复记录的证据。本节参考文件没有时间，不能运行基于 t 的物理 cut；时间选择代码仅说明有有效时间输入时的容器用法。</p><p>按能量排序后同下标配对，是单条响应和清晰能量分离下的初步候选算法，并未覆盖 3.6 的 sharing、共用条及多解事件。保留未匹配结果，再按需要回到完整重建。</p>